# Gemini book embeddings setup

Prepare the database connection and inspect the book and embedding tables before implementing embedding ingestion. This notebook currently checks setup and schema; it does not generate or store embeddings.

## Current implementation scope

The notebook now generates and stores embeddings for a 10-book sample, then checks the inserted records.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from sqlalchemy import create_engine, text, inspect

SUPABASE_DATABASE_URL = os.getenv("SUPABASE_DATABASE_URL")
engine = create_engine(SUPABASE_DATABASE_URL)

## Check database connectivity

Run a lightweight query to confirm the database connection is available.

In [3]:
try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
        print("Connection successful:", result.scalar())

except Exception as e:
    print("Connection failed:", e)

Connection successful: 1


## Register the pgvector type

Teach SQLAlchemy to recognize PostgreSQL `vector` columns so their schema can be inspected correctly.

In [4]:
from pgvector.sqlalchemy import VECTOR
from sqlalchemy.dialects.postgresql.base import ischema_names

ischema_names["vector"] = VECTOR

## Inspect the destination tables

Review the columns, nullability, and foreign keys for `books` and `book_embeddings` before adding embedding records.

In [5]:
inspector = inspect(engine)

for table in ["books", "book_embeddings"]:
    print(f"\n{table.upper()}")

    for column in inspector.get_columns(table):
        print(
            column["name"],
            column["type"],
            "nullable:", column["nullable"]
        )

    print("Foreign keys:", inspector.get_foreign_keys(table))


BOOKS
id BIGINT nullable: False
isbn13 VARCHAR nullable: True
isbn10 VARCHAR nullable: True
title VARCHAR nullable: False
authors TEXT nullable: True
categories TEXT nullable: True
thumbnail TEXT nullable: True
description TEXT nullable: True
published_year INTEGER nullable: True
average_rating DOUBLE PRECISION nullable: True
num_pages INTEGER nullable: True
ratings_count INTEGER nullable: True
title_and_subtitle TEXT nullable: True
tagged_description TEXT nullable: True
simple_categories TEXT nullable: True
anger DOUBLE PRECISION nullable: True
disgust DOUBLE PRECISION nullable: True
fear DOUBLE PRECISION nullable: True
joy DOUBLE PRECISION nullable: True
sadness DOUBLE PRECISION nullable: True
surprise DOUBLE PRECISION nullable: True
neutral DOUBLE PRECISION nullable: True
Foreign keys: []

BOOK_EMBEDDINGS
id BIGINT nullable: False
book_id BIGINT nullable: False
model VARCHAR nullable: False
dimensions INTEGER nullable: False
embedding VECTOR(768) nullable: False
Foreign keys: [{'na

In [6]:
import pandas as pd

## Load book text for processing

Fetch each book ID, title, and description from the database as input for the embedding workflow.

In [7]:
with engine.connect() as conn:
    df_books = pd.read_sql(
        text("SELECT id, title, description FROM books"),
        conn
    )

df_books.sort_values("id").reset_index(drop=True)

,id,title,description
0,1,Gilead,A NOVEL THAT READERS and critics have been eag...
1,2,Spider's Web,A new 'Christie for Christmas' -- a full-lengt...
2,3,Rage of angels,"A memorable, mesmerizing heroine Jennifer -- b..."
3,4,The Four Loves,Lewis' work on the nature of love divides love...
4,5,The Problem of Pain,"""In The Problem of Pain, C.S. Lewis, one of th..."
...,...,...,...
5192,5193,Mistaken Identity,On A Train Journey Home To North India After L...
5193,5194,Journey to the East,This book tells the tale of a man who goes on ...
5194,5195,The Monk Who Sold His Ferrari: A Fable About F...,"Wisdom to Create a Life of Passion, Purpose, a..."
5195,5196,I Am that,This collection of the timeless teachings of o...


## Check for repeated descriptions

Find books that share an identical description so repeated text can be reviewed before embedding.

In [8]:
duplicates = df_books[
    df_books.duplicated(subset=["description"], keep=False)
].sort_values("description")

print(f"Books with duplicated descriptions: {len(duplicates)}")
print(f"Unique duplicated descriptions: {duplicates['description'].nunique()}")

duplicates

Books with duplicated descriptions: 72
Unique duplicated descriptions: 29


,id,title,description
280,275,The Alchemist - Gift Edition,"""My heart is afraid that it will have to suffe..."
370,367,The Alchemist,"""My heart is afraid that it will have to suffe..."
5025,5024,Death,"A tale of music, mortality, friendship and dea..."
4615,4620,Death,"A tale of music, mortality, friendship and dea..."
2579,2572,The Secret Garden,A ten-year-old orphan comes to live in a lonel...
...,...,...,...
4471,4467,The Story of the Life of Lafayette,This is a reproduction of the original artefac...
4098,4089,Arundhati Roy's The God of Small Things,This is part of a new series of guides to cont...
4095,4086,David Foster Wallace's Infinite Jest,This is part of a new series of guides to cont...
3560,3554,Spain,Three-dimensional cutaway illustrations and fl...


## Exclude boilerplate descriptions

Filter out descriptions containing the known reproduction notice to create the pending set for later processing.

In [9]:
boilerplate = "This is a reproduction of the original artefact."

df_pending = df_books[
    ~df_books["description"].str.contains(
        boilerplate,
        case=False,
        na=False,
        regex=False
    )
].copy()

df_pending.sort_values("id").reset_index(drop=True)

,id,title,description
0,1,Gilead,A NOVEL THAT READERS and critics have been eag...
1,2,Spider's Web,A new 'Christie for Christmas' -- a full-lengt...
2,3,Rage of angels,"A memorable, mesmerizing heroine Jennifer -- b..."
3,4,The Four Loves,Lewis' work on the nature of love divides love...
4,5,The Problem of Pain,"""In The Problem of Pain, C.S. Lewis, one of th..."
...,...,...,...
5186,5193,Mistaken Identity,On A Train Journey Home To North India After L...
5187,5194,Journey to the East,This book tells the tale of a man who goes on ...
5188,5195,The Monk Who Sold His Ferrari: A Fable About F...,"Wisdom to Create a Life of Passion, Purpose, a..."
5189,5196,I Am that,This collection of the timeless teachings of o...


## Build embedding documents

Wrap each pending book description as a LangChain `Document` and attach its book ID so a future embedding can be linked back to its source.

In [10]:
from langchain_core.documents import Document

In [11]:
documents = [
    Document(
        page_content=row.description,
        metadata={"book_id": int(row.id)}
    )
    for row in df_pending.itertuples(index=False)
]

print(f"Documents created: {len(documents)}")

Documents created: 5191


In [12]:
documents[0]

Document(metadata={'book_id': 1}, page_content='A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and ac

## Check Gemini credentials

Confirm the API key is configured before making a request to Gemini.

In [13]:
assert os.getenv("GOOGLE_API_KEY"), "Missing GOOGLE_API_KEY"

## Configure the embedding model

Set up Gemini to embed book descriptions for document retrieval, with vectors sized to match the 768-dimension database column.

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

MODEL_NAME = os.getenv("GEMINI_EMBEDDING_MODEL")
EMBEDDING_DIMENSIONS = 768

assert MODEL_NAME, "Missing GEMINI_EMBEDDING_MODEL"

embeddings = GoogleGenerativeAIEmbeddings(
    model=MODEL_NAME,
    task_type="RETRIEVAL_DOCUMENT",
    output_dimensionality=EMBEDDING_DIMENSIONS,
)

## Generate a sample embedding

Embed one book description and inspect its source ID and vector dimensions before processing the full set.

In [15]:
test_vector = embeddings.embed_documents(
    [documents[0].page_content]
)[0]

print("Book ID:", documents[0].metadata["book_id"])
print("Vector dimensions:", len(test_vector))
print("First 5 values:", test_vector[:5])

Book ID: 1
Vector dimensions: 768
First 5 values: [0.024345562, 0.015480793, 0.021809395, -0.014737027, 0.002422453]


## Generate a sample batch

Embed the first 10 prepared descriptions and confirm the number and dimensions of the returned vectors.

In [16]:
batch = documents[:10]
len(batch)

10

In [17]:
texts = [doc.page_content for doc in batch]

vectors = embeddings.embed_documents(texts)

print(f"Embeddings generated: {len(vectors)}")
print(f"Vector dimensions: {len(vectors[0])}")

Embeddings generated: 10
Vector dimensions: 768


In [18]:
for column in inspector.get_columns("book_embeddings"):
    print(
        column["name"],
        column["type"],
        "nullable:", column["nullable"],
        "default:", column["default"]
    )

id BIGINT nullable: False default: nextval('book_embeddings_id_seq'::regclass)
book_id BIGINT nullable: False default: None
model VARCHAR nullable: False default: None
dimensions INTEGER nullable: False default: 768
embedding VECTOR(768) nullable: False default: None


## Reflect the embedding table

Load the existing table definition so inserts use the database column types and constraints.

In [19]:
from sqlalchemy import MetaData, Table

metadata = MetaData()

book_embeddings = Table(
    "book_embeddings",
    metadata,
    autoload_with=engine
)

## Prepare embedding records

Pair each vector with its source book ID, model name, and dimension count.

In [20]:
records = [
    {
        "book_id": doc.metadata["book_id"],
        "model": MODEL_NAME,
        "dimensions": EMBEDDING_DIMENSIONS,
        "embedding": vector
    }
    for doc, vector in zip(batch, vectors, strict=True)
]

## Store the sample embeddings

Insert the prepared records in a database transaction.

In [21]:
with engine.begin() as conn:
    conn.execute(book_embeddings.insert(), records)

print(f"Inserted {len(records)} embeddings.")

Inserted 10 embeddings.


## Verify stored embeddings

Query the latest records for this model to confirm the sample was saved.

In [24]:
with engine.connect() as conn:
    result = pd.read_sql(
        text("""
            SELECT book_id, model, dimensions
            FROM book_embeddings
            WHERE model = :model
            ORDER BY id DESC
            LIMIT 10
        """),
        conn,
        params={"model": MODEL_NAME}
    )

result

,book_id,model,dimensions
0,9,gemini-embedding-2,768
1,8,gemini-embedding-2,768
2,7,gemini-embedding-2,768
3,2489,gemini-embedding-2,768
4,6,gemini-embedding-2,768
5,5,gemini-embedding-2,768
6,4,gemini-embedding-2,768
7,3,gemini-embedding-2,768
8,2,gemini-embedding-2,768
9,1,gemini-embedding-2,768
